# 19 · Benchmarking and Model Evaluation

Standardised evaluation using EarthNets methodology.

## Contents
1. EarthNets dataset selection
2. Benchmark configuration
3. Cross-model comparison
4. Leaderboard
5. Visualisation

## 1 · EarthNets Dataset Selection

In [ ]:
from pygeovision.datasets.benchmark import BenchmarkBuilder
builder = BenchmarkBuilder()
builder.print_all(n=5)

## 2 · Model Comparison

In [ ]:
from pygeovision.benchmark.evaluator import BenchmarkResult, ModelEvaluator

RESULTS = [
    BenchmarkResult('SegFormer-B5',        'LoveDA','segmentation', mean_iou=0.785, mean_f1=0.862, accuracy=0.941, inference_ms_per_image=23.4),
    BenchmarkResult('UNet-EfficientNet-B4','LoveDA','segmentation', mean_iou=0.762, mean_f1=0.843, accuracy=0.928, inference_ms_per_image=12.8),
    BenchmarkResult('SegFormer-B2',        'LoveDA','segmentation', mean_iou=0.756, mean_f1=0.838, accuracy=0.929, inference_ms_per_image=8.1),
    BenchmarkResult('DeepLabV3+-R101',     'LoveDA','segmentation', mean_iou=0.748, mean_f1=0.832, accuracy=0.921, inference_ms_per_image=31.2),
    BenchmarkResult('UNet-ResNet50',       'LoveDA','segmentation', mean_iou=0.731, mean_f1=0.817, accuracy=0.914, inference_ms_per_image=9.8),
    BenchmarkResult('Fast-SCNN',           'LoveDA','segmentation', mean_iou=0.673, mean_f1=0.761, accuracy=0.886, inference_ms_per_image=2.4),
]
RESULTS.sort(key=lambda r: r.primary_metric, reverse=True)
ev = ModelEvaluator(task='segmentation')
ev.print_leaderboard(RESULTS)

  Leaderboard: LoveDA | Task: segmentation
  #   Model                              mIoU         F1        Acc    ms/img
  ─────────────────────────────────────────────────────────────────────────────────
  1   SegFormer-B5                      0.7850     0.8620     0.9410       23.4
  2   UNet-EfficientNet-B4              0.7620     0.8430     0.9280       12.8
  3   SegFormer-B2                      0.7560     0.8380     0.9290        8.1
  4   DeepLabV3+-R101                   0.7480     0.8320     0.9210       31.2
  5   UNet-ResNet50                     0.7310     0.8170     0.9140        9.8
  6   Fast-SCNN                         0.6730     0.7610     0.8860        2.4


## 3 · Speed-Accuracy Trade-off

In [ ]:
import matplotlib.pyplot as plt, numpy as np
models  = [r.model_name for r in RESULTS]
mious   = [r.mean_iou for r in RESULTS]
speeds  = [r.inference_ms_per_image for r in RESULTS]
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(speeds, mious, s=150, c=range(len(models)), cmap='RdYlGn_r', zorder=5)
for name, sp, mi in zip(models, speeds, mious):
    ax.annotate(name, (sp, mi), xytext=(8, -4), textcoords='offset points', fontsize=9)
ax.axhline(0.75, linestyle='--', alpha=0.4, color='green', label='IoU=0.75')
ax.axvline(10,   linestyle='--', alpha=0.4, color='orange', label='10ms real-time')
ax.set_xlabel('Inference Time (ms/image)', fontsize=13)
ax.set_ylabel('Mean IoU', fontsize=13)
ax.set_title('Speed vs Accuracy - LoveDA Segmentation', fontsize=14, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
ax.set_xlim(0, 38); ax.set_ylim(0.64, 0.82)
plt.tight_layout()
import os; os.makedirs('./results/benchmark', exist_ok=True)
plt.savefig('./results/benchmark/speed_accuracy.png', dpi=150)
plt.show()
print(f'Best accuracy:   SegFormer-B5 (mIoU={max(mious):.3f})')
print(f'Fastest:         Fast-SCNN    ({min(speeds):.1f}ms/img)')
print(f'Best trade-off:  SegFormer-B2 (mIoU=0.756, 8.1ms)')

## 4 · Persistent Leaderboard

In [ ]:
from pygeovision.benchmark import Leaderboard
lb = Leaderboard('./results/benchmark/leaderboard.json')
for r in RESULTS:
    lb.add(r)
lb.print(task='segmentation', dataset='LoveDA')
lb.export_csv('./results/benchmark/leaderboard.csv')
print('Leaderboard exported to CSV')

## 5 · Recommended Benchmark Configuration

In [ ]:
from pygeovision.datasets.benchmark import BenchmarkBuilder, BENCHMARK_TASKS

builder = BenchmarkBuilder()
rec = builder.recommended_for_paper('segmentation')
print(f'Recommended benchmark for segmentation paper:')
print(f'  Primary metric: {rec["primary_metric"]}')
print(f'  Split:          {rec["split"]}')
print(f'  Datasets ({len(rec["datasets"])}):')
for d in rec['datasets']:
    print(f'    - {d["name"]:<30} {d["year"]}  {d["resolution_m"]}m  {d["n_samples"]:,} samples')
print()
print('Save benchmark configs:')
print("  saved = builder.save_all('./benchmarks/', n=5)")
print("  # Saves JSON configs for all tasks")